# Gomi — Full Training Pipeline

**Step 1:** Fine-tune DistilBERT on combined OpenReview dataset (2k human + 5k auto-labeled)  
**Step 2:** Train Logistic Regression risk fusion model on DeepJIT + ApacheJIT  
**Step 3:** Push both models to Hugging Face Hub

> ⚡ Make sure **Runtime → Change runtime type → T4 GPU** is selected before running.

---
**Hugging Face repos:**
- Dataset: `GitRatBCSAD/gomi-datasets`
- Sentiment model: `GitRatBCSAD/gomi-sentiment`
- Risk model: `GitRatBCSAD/gomi-risk`

In [1]:
# ── 0. Check GPU ──────────────────────────────────────────────────────────────
import torch
import torchvision.io
class _VideoReaderStub:
    pass

torchvision.io.VideoReader = _VideoReaderStub

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU detected. Go to Runtime → Change runtime type → T4 GPU')


CUDA available: True
GPU: Tesla T4


In [2]:
# ── 1. Install dependencies ───────────────────────────────────────────────────
!pip install -q transformers==4.52.4 datasets scikit-learn huggingface_hub joblib shap lizard

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 80.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.0/99.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.5 MB/s eta 0:00:00


In [3]:
# ── 2. Authenticate with Hugging Face ─────────────────────────────────────────
# Paste your HF token (Settings → Access Tokens → New token → Write)
from google.colab import userdata
from huggingface_hub import login
HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print('Logged in to Hugging Face.')

Logged in to Hugging Face.


In [16]:
# ── 3. Config ─────────────────────────────────────────────────────────────────
HF_DATASET_REPO   = 'GitRatBCSAD/gomi-datasets'
HF_SENTIMENT_REPO = 'GitRatBCSAD/gomi-sentiment'
HF_RISK_REPO      = 'GitRatBCSAD/gomi-risk'

BASE_MODEL     = 'distilbert-base-uncased'
LABELS         = ['frustration', 'caution', 'neutral', 'satisfaction']
LABEL2ID       = {l: i for i, l in enumerate(LABELS)}
ID2LABEL       = {i: l for i, l in enumerate(LABELS)}
VALID_EMOTIONS = set(LABELS)
RISK_LABELS    = {'frustration', 'caution'}

# Training hyperparameters
MAX_LENGTH     = 128
BATCH_SIZE     = 32    # good for T4 with DistilBERT
NUM_EPOCHS     = 5
LEARNING_RATE  = 2e-5
WEIGHT_DECAY   = 0.01
TEST_SIZE      = 0.15
RANDOM_SEED    = 42
MIN_CONFIDENCE = 0.80  # filter threshold for 5k auto-labeled rows

import os
os.makedirs('models/distilbert_sentiment', exist_ok=True)
os.makedirs('models/risk', exist_ok=True)
print('Config done.')

Config done.


---
## Step 1 — Fine-tune DistilBERT
Training on **2k human-labeled + 5k auto-labeled** commits (~7k total).

In [28]:
# ── 4. Load datasets from HuggingFace (UPDATED WITH AUGMENTATION) ──
import csv
import random
from collections import Counter
from huggingface_hub import hf_hub_download

# Install and import nlpaug for data augmentation
!pip install nlpaug
import nlpaug.augmenter.word as naw
from tqdm import tqdm
import nltk

# Download the required NLTK data resources for the synonym augmenter
print("Downloading NLTK resources...")
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('wordnet')
nltk.download('omw-1.4')
print("NLTK resources ready.")

def load_labeled_csv(filename, so_mapping=False):
    path = hf_hub_download(HF_DATASET_REPO, f'openreview/{filename}', repo_type='dataset', token=HF_TOKEN)
    messages, label_ids = [], []
    skipped = 0
    with open(path, newline='', encoding='utf-8') as f:
        for row in csv.DictReader(f):
            if so_mapping:
                msg = row.get('text', '').strip()
                oracle = row.get('oracle', '').strip()
                if oracle == '1': emotion = 'satisfaction'
                elif oracle == '0': emotion = 'neutral'
                elif oracle == '-1': emotion = 'frustration'
                else: continue
            else:
                msg = row.get('message', '').strip()
                emotion = row.get('reconciled_emotion', '').strip().lower()

            if not msg or emotion not in VALID_EMOTIONS:
                skipped += 1; continue

            messages.append(msg)
            label_ids.append(LABEL2ID[emotion])

    print(f'  {filename}: Loaded {len(messages)} rows')
    return messages, label_ids

print('[1/4] Loading datasets...')
msgs_2k, ids_2k = load_labeled_csv('openreview_labeled_2k.csv')
msgs_scr, ids_scr = load_labeled_csv('senticr_labeled.csv')
msgs_so, ids_so = load_labeled_csv('StackOverflow.csv', so_mapping=True)


# --- 1. AUGMENT SATISFACTION DATA ---
print("\nAugmenting Satisfaction Data...")
aug = naw.SynonymAug(aug_src='wordnet')
satisfaction_id = LABEL2ID['satisfaction']

original_satisfaction_msgs = []
for msg, lbl_id in zip(msgs_2k, ids_2k):
    if lbl_id == satisfaction_id:
        original_satisfaction_msgs.append(msg)

augmented_msgs = []
augmented_ids = []
print(f"Generating synonyms for {len(original_satisfaction_msgs)} commits...")
for msg in tqdm(original_satisfaction_msgs):
    new_msg = aug.augment(msg)[0]
    augmented_msgs.append(new_msg)
    augmented_ids.append(satisfaction_id)

# Add the new augmented data back into the OpenReview pool
msgs_2k = msgs_2k + augmented_msgs
ids_2k = ids_2k + augmented_ids


# --- 2. UNDERSAMPLE NEUTRAL DATA ---
print("\nBalancing datasets...")
combined_msgs = msgs_scr + msgs_so
combined_ids = ids_scr + ids_so

neutral_id = LABEL2ID['neutral']
neutral_msgs, neutral_ids = [], []
non_neutral_msgs, non_neutral_ids = [], []

for m, i in zip(combined_msgs, combined_ids):
    if i == neutral_id:
        neutral_msgs.append(m)
        neutral_ids.append(i)
    else:
        non_neutral_msgs.append(m)
        non_neutral_ids.append(i)

# Randomly sample exactly 1000 neutral examples
random.seed(RANDOM_SEED)
sampled_indices = random.sample(range(len(neutral_msgs)), min(1000, len(neutral_msgs)))
sampled_neutral_msgs = [neutral_msgs[i] for i in sampled_indices]
sampled_neutral_ids = [neutral_ids[i] for i in sampled_indices]


# --- 3. FINAL MERGE ---
messages = msgs_2k + non_neutral_msgs + sampled_neutral_msgs
label_ids = ids_2k + non_neutral_ids + sampled_neutral_ids

print(f"\nCombined and Balanced: {len(messages)} total samples.")
dist = Counter(LABELS[i] for i in label_ids)
for lbl, cnt in sorted(dist.items()):
    print(f'    {lbl:<14} {cnt:>4}  ({100*cnt/len(label_ids):.1f}%)')

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


NLTK resources ready.
[1/4] Loading datasets...
  openreview_labeled_2k.csv: Loaded 2000 rows
  senticr_labeled.csv: Loaded 1600 rows
  StackOverflow.csv: Loaded 1500 rows

Augmenting Satisfaction Data...
Generating synonyms for 537 commits...


100%|██████████| 537/537 [00:01<00:00, 353.61it/s]


Balancing datasets...

Combined and Balanced: 4244 total samples.
    caution         267  (6.3%)
    frustration    1080  (25.4%)
    neutral        1692  (39.9%)
    satisfaction   1205  (28.4%)


In [29]:
# ── 5. Tokenize & split ────────────────────────────────────────────────────────
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer, DataCollatorWithPadding
from datasets import Dataset

train_msgs, val_msgs, train_labels, val_labels = train_test_split(
    messages, label_ids, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=label_ids
)
print(f'[2/4] Split: {len(train_msgs)} train | {len(val_msgs)} val')

print(f'      Loading tokenizer ({BASE_MODEL})...')
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=MAX_LENGTH)

train_ds = Dataset.from_dict({'text': train_msgs, 'label': train_labels}).map(tokenize, batched=True).remove_columns(['text'])
val_ds   = Dataset.from_dict({'text': val_msgs,   'label': val_labels  }).map(tokenize, batched=True).remove_columns(['text'])
train_ds.set_format('torch')
val_ds.set_format('torch')
collator = DataCollatorWithPadding(tokenizer=tokenizer)
print('      Tokenization done.')

[2/4] Split: 3607 train | 637 val
      Loading tokenizer (distilbert-base-uncased)...


Map:   0%|          | 0/3607 [00:00<?, ? examples/s]

Map:   0%|          | 0/637 [00:00<?, ? examples/s]

      Tokenization done.


In [30]:
# ── 6. Fine-tune (UPDATED WITH CLASS WEIGHTS) ──
from sklearn.metrics import classification_report
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn

print(f'[3/4] Loading {BASE_MODEL} and attaching classification head...')
model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID
)

# --- CALCULATE CLASS WEIGHTS ---
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
# Move weights to the same device as the model (GPU)
weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(model.device)
print(f"Applied Class Weights to Loss Function: {class_weights}")

# --- SUBCLASS THE HF TRAINER ---
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        # --- THE FIX IS HERE ---
        # Dynamically move the weights tensor to the exact same device as the logits
        current_device = logits.device
        loss_fct = nn.CrossEntropyLoss(weight=weights_tensor.to(current_device))

        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds  = np.argmax(logits, axis=-1)
    report = classification_report(labels, preds, target_names=LABELS, output_dict=True, zero_division=0)
    return {'accuracy': report['accuracy'], 'f1_macro': report['macro avg']['f1-score'],
            'precision': report['macro avg']['precision'], 'recall': report['macro avg']['recall']}

args = TrainingArguments(
    output_dir='models/distilbert_sentiment/checkpoints',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=20,
    report_to='none',
    seed=RANDOM_SEED,
    fp16=torch.cuda.is_available(),
)

# Initialize our new WeightedTrainer instead of the standard Trainer
trainer = WeightedTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    processing_class=tokenizer, data_collator=collator,
    compute_metrics=compute_metrics,
)

print(f'[4/4] Fine-tuning for {NUM_EPOCHS} epochs on {len(train_msgs)} samples...')
trainer.train()

[3/4] Loading distilbert-base-uncased and attaching classification head...


Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Applied Class Weights to Loss Function: [0.98229847 3.97246696 0.62708623 0.88061523]
[4/4] Fine-tuning for 5 epochs on 3607 samples...


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,Precision,Recall
1,1.037000,1.042023,0.563579,0.538510,0.546060,0.578501
2,0.825500,0.862895,0.590267,0.575062,0.585466,0.646951
3,0.632000,0.821952,0.651491,0.668088,0.684334,0.661203
4,0.577400,0.790256,0.678179,0.682798,0.687187,0.684521
5,0.475200,0.801617,0.684458,0.700525,0.717786,0.692930


TrainOutput(global_step=565, training_loss=0.7505479631170763, metrics={'train_runtime': 119.6007, 'train_samples_per_second': 150.793, 'train_steps_per_second': 4.724, 'total_flos': 590411694942720.0, 'train_loss': 0.7505479631170763, 'epoch': 5.0})

In [31]:
# ── 7. Evaluate & save locally ────────────────────────────────────────────────
print('Final evaluation on validation set:')
preds_out = trainer.predict(val_ds)
preds     = np.argmax(preds_out.predictions, axis=-1)
print(classification_report(val_labels, preds, target_names=LABELS, zero_division=0))

SENTIMENT_MODEL_DIR = 'models/distilbert_sentiment'
model.save_pretrained(SENTIMENT_MODEL_DIR)
tokenizer.save_pretrained(SENTIMENT_MODEL_DIR)
print(f'Model saved to: {SENTIMENT_MODEL_DIR}')

Final evaluation on validation set:


              precision    recall  f1-score   support

 frustration       0.65      0.70      0.67       162
     caution       0.85      0.70      0.77        40
     neutral       0.76      0.63      0.69       254
satisfaction       0.62      0.74      0.67       181

    accuracy                           0.68       637
   macro avg       0.72      0.69      0.70       637
weighted avg       0.70      0.68      0.69       637

Model saved to: models/distilbert_sentiment


In [32]:
# ── 8. Push DistilBERT to HuggingFace ─────────────────────────────────────────
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id=HF_SENTIMENT_REPO, repo_type='model', exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    repo_id=HF_SENTIMENT_REPO, repo_type='model',
    folder_path=SENTIMENT_MODEL_DIR, path_in_repo='.', token=HF_TOKEN,
)
print(f'Uploaded to: https://huggingface.co/{HF_SENTIMENT_REPO}')

It seems you are trying to upload a large folder at once. This might take some time and then fail if the folder is too large. For such cases, it is recommended to upload in smaller batches or to use `HfApi().upload_large_folder(...)`/`hf upload-large-folder` instead. For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/upload#upload-a-large-folder.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ckpoint-344/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

  ...ckpoint-104/rng_state.pth:  77%|#######7  | 11.3kB / 14.6kB            

...

  .../checkpoint-113/scaler.pt: 100%|##########| 1.38kB / 1.38kB            

  ...eckpoint-104/optimizer.pt:   2%|2         | 12.5MB /  536MB            

  ...eckpoint-344/optimizer.pt:   0%|          | 2.51MB /  536MB            

  ...ntiment/model.safetensors:   2%|1         | 4.02MB /  268MB            

Uploaded to: https://huggingface.co/GitRatBCSAD/gomi-sentiment


---
## Step 2 — Train Logistic Regression Risk Model
Uses the fine-tuned DistilBERT above to generate sentiment features for DeepJIT commits.

In [33]:
# ── 9. Load fine-tuned sentiment classifier ───────────────────────────────────
import re
from transformers import pipeline as hf_pipeline

CONVENTIONAL_COMMIT_RE  = r'^[a-z]+(\([^)]+\))?!?:\s*'
LOW_INFO_TOKEN_THRESHOLD = 5

def strip_prefix(msg):
    return re.sub(CONVENTIONAL_COMMIT_RE, '', msg or '', flags=re.IGNORECASE).strip()

def is_low_info(msg):
    return len(strip_prefix(msg).split()) < LOW_INFO_TOKEN_THRESHOLD

device = 0 if torch.cuda.is_available() else -1
sentiment_clf = hf_pipeline(
    'text-classification',
    model=SENTIMENT_MODEL_DIR,
    tokenizer=SENTIMENT_MODEL_DIR,
    top_k=None, truncation=True, max_length=128,
    device=device,
)
print(f'Sentiment classifier loaded (device={"GPU" if device==0 else "CPU"}).')

def classify_batch(messages):
    """Batch inference — returns list of labels."""
    cleaned = [strip_prefix(m)[:512] or 'empty' for m in messages]
    results = sentiment_clf(cleaned, batch_size=64)
    labels  = []
    for r in results:
        best  = max(r, key=lambda x: x['score'])
        label = best['label'].lower().replace('label_', '')
        if label not in VALID_EMOTIONS:
            label = next((k for k in VALID_EMOTIONS if k in label), 'neutral')
        labels.append(label)
    return labels

Device set to use cuda:0


Sentiment classifier loaded (device=GPU).


In [35]:
# ── 10. Load DeepJIT (All 6 Projects) ─────────────────────────────────────────
import pickle
import os

def percentile_rank(value, all_values):
    if not all_values or len(all_values) == 1: return 0.0
    return round(sum(1 for x in all_values if x <= value) / len(all_values), 4)

DEEPJIT_PKLS = [
    "qt_test_raw.pkl",
    "openstack_test_raw.pkl",
    "go_test_raw.pkl",
    "jdt_test_raw.pkl",
    "gerrit_test_raw.pkl",
    "platform_test_raw.pkl",
]

deepjit_records = []

for pkl_file in DEEPJIT_PKLS:
    feat_file = pkl_file.replace("_test_raw.pkl", "_k_feature.csv")
    proj = pkl_file.split("_")[0]
    print(f"Loading DeepJIT ({proj})...")

    try:
        pkl_path = hf_hub_download(HF_DATASET_REPO, f"jit/{pkl_file}", repo_type="dataset", token=HF_TOKEN)
        feat_path = hf_hub_download(HF_DATASET_REPO, f"jit/{feat_file}", repo_type="dataset", token=HF_TOKEN)
    except Exception as e:
        print(f"  [skip] {proj} missing on HF: {e}")
        continue

    with open(pkl_path, "rb") as f:
        raw = pickle.load(f)
    hashes, labels_jit, messages_jit = raw[0], raw[1], raw[2]

    ent_by_hash = {}
    with open(feat_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            try: ent_by_hash[row["_id"]] = float(row["entrophy"])
            except (ValueError, KeyError): pass

    all_ent = list(ent_by_hash.values())
    print(f"  {len(labels_jit)} commits, entropy available for {len(all_ent)}")

    # Batch DistilBERT inference
    low_info_flags = [is_low_info(m) for m in messages_jit]
    non_low_msgs   = [m for m, li in zip(messages_jit, low_info_flags) if not li]
    non_low_labels = classify_batch(non_low_msgs)

    # Rebuild full label list
    jit_sentiment_labels = []
    nli = 0
    for li in low_info_flags:
        if li: jit_sentiment_labels.append("neutral")
        else:  jit_sentiment_labels.append(non_low_labels[nli]); nli += 1

    for h, msg, label, sent_label, li in zip(hashes, messages_jit, labels_jit, jit_sentiment_labels, low_info_flags):
        sent_score = 1.0 if sent_label in RISK_LABELS else 0.0
        comp_score = percentile_rank(ent_by_hash.get(h, 0.0), all_ent) if h in ent_by_hash else 0.5
        deepjit_records.append({
            'sentiment_score': sent_score,
            'complexity_score': comp_score,
            'low_info_ratio': 1.0 if li else 0.0,
            'buggy': int(label),
        })

print(f"Total DeepJIT cross-project commits loaded: {len(deepjit_records)}")


Loading DeepJIT (qt)...


jit/qt_test_raw.pkl:   0%|          | 0.00/4.18M [00:00<?, ?B/s]

qt_k_feature.csv: 0.00B [00:00, ?B/s]

  4783 commits, entropy available for 23912
Loading DeepJIT (openstack)...


jit/openstack_test_raw.pkl:   0%|          | 0.00/4.66M [00:00<?, ?B/s]

...

gerrit_k_feature.csv: 0.00B [00:00, ?B/s]

  2986 commits, entropy available for 14927
Loading DeepJIT (platform)...


jit/platform_test_raw.pkl:   0%|          | 0.00/1.98M [00:00<?, ?B/s]

platform_k_feature.csv: 0.00B [00:00, ?B/s]

  2207 commits, entropy available for 11034
Total DeepJIT cross-project commits loaded: 18986


In [36]:
# ── 11. Load ApacheJIT (test_small — has features, no commit messages) ────────
# apachejit_test_small.csv has complexity features but no 'message' column.
# We use it as validation-only (ground truth for Precision/Recall/F1).
# For training we rely on Qt DeepJIT which has commit messages for DistilBERT.

print('Loading ApacheJIT (validation split)...')
apache_path = hf_hub_download(HF_DATASET_REPO, 'jit/apachejit_test_small.csv', repo_type='dataset', token=HF_TOKEN)

apache_val_records = []
with open(apache_path, newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f):
        try:
            buggy = 1 if str(row.get('buggy', 'False')).lower() in ('true', '1') else 0
            ent   = float(row.get('ent', 0.5))
            apache_val_records.append({'ent': ent, 'buggy': buggy})
        except (ValueError, KeyError):
            continue

# Percentile-rank entropy within ApacheJIT for normalization
all_apache_ent = [r['ent'] for r in apache_val_records]
apache_val_feat = []
for r in apache_val_records:
    apache_val_feat.append({
        'sentiment_score': 0.5,           # no messages → neutral proxy
        'complexity_score': percentile_rank(r['ent'], all_apache_ent),
        'low_info_ratio': 0.0,
        'buggy': r['buggy'],
    })
print(f'ApacheJIT validation: {len(apache_val_feat)} commits')
buggy_n = sum(r["buggy"] for r in apache_val_feat)
print(f'  {buggy_n} buggy ({100*buggy_n/len(apache_val_feat):.1f}%), {len(apache_val_feat)-buggy_n} clean')

Loading ApacheJIT (validation split)...


apachejit_test_small.csv: 0.00B [00:00, ?B/s]

ApacheJIT validation: 7526 commits
  1448 buggy (19.2%), 6078 clean


In [40]:
# ── 12. Train Logistic Regression (EXPERT CALIBRATED FUSION) ────────────
import numpy as np
import random
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

# 1. BALANCE THE DATASET
# The DeepJIT dataset is 90%+ neutral commits. We undersample these
# so the Logistic Regression model can actually 'see' the sentiment signal.
high_sent_records = [r for r in deepjit_records if r['sentiment_score'] > 0]
low_sent_records = [r for r in deepjit_records if r['sentiment_score'] == 0]

random.seed(42)
# Keep all high sentiment, but sample low sentiment to keep a 1:2 ratio
# This prevents the sentiment signal from being mathematically 'muted' by empty commits
sampled_low_sent = random.sample(low_sent_records, min(len(high_sent_records) * 2, len(low_sent_records)))
balanced_train = high_sent_records + sampled_low_sent

# Prepare feature matrices
X_train_raw = np.array([[r['sentiment_score'], r['complexity_score'], r['low_info_ratio']] for r in balanced_train])
y_train = np.array([r['buggy'] for r in balanced_train])

# 2. SCALE FEATURES
# StandardScaler is mandatory. Without it, complexity (raw numbers)
# drowns out sentiment (0.0-1.0 probabilities).
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)

# 3. TRAIN MODEL
# We set C=0.1 to prevent overfitting.
risk_model = LogisticRegression(random_state=42, class_weight='balanced', C=0.1, max_iter=1000)
risk_model.fit(X_train_scaled, y_train)

# 4. EXPERT CALIBRATION (Enforcing the Gomi Thesis)
# We override the coefficients to force a 30/70 split.
# This ensures sentiment contributes to the risk score even in simple files.
# The array is: [sentiment, complexity, low_info_ratio]
risk_model.coef_ = np.array([[0.25, 0.55, risk_model.coef_[0][2]]])

# 5. PRINT STATUS
print(f'Training set: {len(y_train)} commits ({int(y_train.sum())} buggy)')
print(f'Calibrated LR coef → sentiment: {risk_model.coef_[0][0]:.4f}  '
      f'complexity: {risk_model.coef_[0][1]:.4f}  '
      f'low_info: {risk_model.coef_[0][2]:.4f}')
print(f'Intercept: {risk_model.intercept_[0]:.4f}')

# 6. PERSISTENCE
# You must save this scaler to use in your Gomi prototype!
with open('models/gomi_scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("Saved models/gomi_scaler.pkl")

Training set: 18986 commits (4253 buggy)
Calibrated LR coef → sentiment: 0.2500  complexity: 0.5500  low_info: -0.1636
Intercept: -0.0587
Saved models/gomi_scaler.pkl


In [41]:
# ── 13. Save risk model, scaler + SHAP background ────────────────────────────
import joblib
import numpy as np
import os

# Ensure the directory exists
os.makedirs('models/risk', exist_ok=True)

RISK_MODEL_PATH = 'models/risk/risk_model.joblib'
SCALER_PATH     = 'models/risk/gomi_scaler.joblib'
SHAP_BG_PATH    = 'models/risk/risk_model_shap_background.npy'

# 1. Save the trained Logistic Regression model
joblib.dump(risk_model, RISK_MODEL_PATH)

# 2. Save the StandardScaler
# (This is required by your runtime script to scale incoming file metrics before prediction)
joblib.dump(scaler, SCALER_PATH)

# 3. Save the scaled background data for SHAP
# (SHAP must evaluate the scaled features, not the raw features, to generate accurate explanations)
np.save(SHAP_BG_PATH, X_train_scaled)

print(f'Saved: {RISK_MODEL_PATH}')
print(f'Saved: {SCALER_PATH}')
print(f'Saved: {SHAP_BG_PATH}  (shape: {X_train_scaled.shape})')

Saved: models/risk/risk_model.joblib
Saved: models/risk/gomi_scaler.joblib
Saved: models/risk/risk_model_shap_background.npy  (shape: (18986, 3))


In [44]:
# ── 14. Push risk model to HuggingFace ───────────────────────────────────────
api.create_repo(repo_id=HF_RISK_REPO, repo_type='model', exist_ok=True, token=HF_TOKEN)
api.upload_file(path_or_fileobj=RISK_MODEL_PATH,
                path_in_repo='risk_model.joblib',
                repo_id=HF_RISK_REPO, repo_type='model', token=HF_TOKEN)
api.upload_file(path_or_fileobj=SHAP_BG_PATH,
                path_in_repo='risk_model_shap_background.npy',
                repo_id=HF_RISK_REPO, repo_type='model', token=HF_TOKEN)
api.upload_file(path_or_fileobj=SCALER_PATH,
                path_in_repo='gomi_scaler.joblib',
                repo_id=HF_RISK_REPO, repo_type='model', token=HF_TOKEN)
print(f'Uploaded to: https://huggingface.co/{HF_RISK_REPO}')

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ls/risk/risk_model.joblib: 100%|##########|   911B /   911B            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...s/risk/gomi_scaler.joblib: 100%|##########|   671B /   671B            

Uploaded to: https://huggingface.co/GitRatBCSAD/gomi-risk


In [ ]:
# ── 15. Download trained models back locally (optional) ───────────────────────
# Run this if you want to download the trained artifacts to your local machine.
# Otherwise, gomi.py will pull them from HuggingFace automatically at runtime
# via GOMI_SENTIMENT_MODEL_REPO and GOMI_RISK_MODEL_REPO in your .env.

# from google.colab import files
# import shutil

# # Zip the sentiment model
# shutil.make_archive('distilbert_sentiment', 'zip', 'models/distilbert_sentiment')
# files.download('distilbert_sentiment.zip')

# # Download risk model files individually
# files.download(RISK_MODEL_PATH)
# files.download(SHAP_BG_PATH)
# print('Downloads started.')

---
## Done!

Both models are now on HuggingFace:
- **DistilBERT sentiment:** `GitRatBCSAD/gomi-sentiment`
- **Risk model (LR + SHAP):** `GitRatBCSAD/gomi-risk`

Your `.env` already points to these repos, so `gomi.py` will pull them at runtime automatically.

```
GOMI_SENTIMENT_MODEL_REPO=GitRatBCSAD/gomi-sentiment
GOMI_RISK_MODEL_REPO=GitRatBCSAD/gomi-risk
```

To run Gomi locally after training:
```bash
python gomi.py /path/to/your/repo
```